In [4]:
import os

import kaggle
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
import cv2

import torch
import torch.nn as nn
from torch.nn.functional import relu
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms

import segmentation_models_pytorch as smp
import import_ipynb

from torch.optim import Adam

device = torch.device('cuda')

from sklearn.model_selection import train_test_split

from tqdm import tqdm
import os
import gc
import logging
from datetime import datetime as dt
import copy
import time

# Import our own custom implementations of the models we used
import models as md

logging.basicConfig(level=logging.INFO, format='%(levelname)s - %(asctime)s - %(message)s')

num_epochs = 200

num_workers = 1
patience = 8 #if use_rotation or use_reflection else 8
combined_loss_weight = 0
tol_neg_sig = 2

train_model = True

model_dir = os.path.join('models')
os.makedirs(model_dir, exist_ok=True)

os.makedirs('output', exist_ok=True)
results_fp = os.path.join('output', 'results.parquet')
timing_fp = os.path.join('output', 'timing.parquet')
if not os.path.exists(results_fp):
    pd.DataFrame().to_parquet(results_fp)
if not os.path.exists(timing_fp):
    pd.DataFrame().to_parquet(timing_fp)


class DataPreprocessing:

    base_dir = os.path.join('data', 'USA_segmentation')
    mask_dir = os.path.join(base_dir, 'masks')
    nrg_dir = os.path.join(base_dir, 'NRG_images')
    rgb_dir = os.path.join(base_dir, 'RGB_images')

    seed = 9999
    test_size = 0.2
    val_size = 0.1

    def __init__(self, model_name, size_std_strat, size_std_dim, use_rotation, use_reflection):

        self.use_rotation = use_rotation
        self.use_reflection = use_reflection
        self.size_std_strat = size_std_strat
        self.size_std_dim = size_std_dim

        metadata_df = pd.DataFrame([x[5:].removesuffix('.png') for x in os.listdir(DataPreprocessing.mask_dir)], columns=['raw_key'])
        metadata_df['state'] = metadata_df['raw_key'].str.slice(stop=2)
        raw_masks_list = []
        masks_list = []
        nrg_list = []
        rgb_list = []
        shapes = []
        target_pixel_count = []
        raw_target_pixel_count = []
        tiles = []

        for ii in range(metadata_df.shape[0]):
            key = metadata_df.raw_key.iloc[ii]

            raw_mask = cv2.imread(os.path.join(DataPreprocessing.mask_dir, f'mask_{key}.png'), cv2.IMREAD_GRAYSCALE)
            raw_masks_list.append(raw_mask)
            raw_bgr = cv2.imread(os.path.join(DataPreprocessing.rgb_dir, f'RGB_{key}.png'))
            # Convert BGR to RGB
            rgb = cv2.cvtColor(raw_bgr, cv2.COLOR_BGR2RGB)
            raw_nrg = cv2.imread(os.path.join(DataPreprocessing.nrg_dir, f'NRG_{key}.png'))

            if size_std_strat == 'zero_padding':
                padded_mask, padded_rgb, padded_nrg = md.add_zero_padding(raw_mask, rgb, raw_nrg, size_std_dim)

            elif size_std_strat == 'use_resize':
                padded_mask, padded_rgb, padded_nrg = md.resize_img(raw_mask, rgb, raw_nrg, size_std_dim)

            elif size_std_strat == 'use_patching':
                padded_mask, padded_rgb, padded_nrg = md.split_patches(raw_mask, rgb, raw_nrg, size_std_dim)

            else:
                padded_mask, padded_rgb, padded_nrg = [raw_mask], [rgb], [raw_nrg]

            masks_list.extend(padded_mask)
            rgb_list.extend(padded_rgb)
            nrg_list.extend(padded_nrg)

            img_shapes = set([masks_list[-1].shape, nrg_list[-1].shape[:-1], rgb_list[-1].shape[:-1]])

            if len(img_shapes) > 1: raise ValueError(f'Image shapes mismatch, key: {key}')

            shapes.append(raw_mask.shape)
            raw_target_pixel_count.append(raw_mask.sum() / 255.0)
            target_pixel_count.append(padded_mask[0].sum() / 255.0)
            tiles.append(len(padded_mask))

        metadata_df[['height', 'width']] = shapes
        metadata_df['raw_num_target_pixel'] = raw_target_pixel_count
        metadata_df['num_target_pixel'] = target_pixel_count
        metadata_df['target_pixel_ratio'] = metadata_df['raw_num_target_pixel'] / (metadata_df['height'] * metadata_df['width'])
        metadata_df['num_tiles'] = tiles
        states = [state for state, count in metadata_df[['state', 'num_tiles']].values for _ in range(count)]

        full_train_idx, test_idx = train_test_split(np.arange(len(masks_list)), stratify=states, test_size=DataPreprocessing.test_size, random_state=DataPreprocessing.seed)
        train_idx, val_idx = train_test_split(full_train_idx, stratify=[states[ii] for ii in full_train_idx], test_size=DataPreprocessing.val_size, random_state=DataPreprocessing.seed)

        if size_std_strat != '':
            y = np.stack(masks_list).astype(np.float32) / 255.0
            if model_name == 'ada-net-adapt':
                X = np.stack(rgb_list).astype(np.float32) / 255.0
            else:
                X = np.stack([np.concat([rgb, nrg], axis=2) for rgb, nrg in zip(rgb_list, nrg_list)]).astype(np.float32) / 255.0
            X_train, X_val, self.X_test, y_train, y_val, self.y_test = X[train_idx], X[val_idx], X[test_idx], y[train_idx], y[val_idx], y[test_idx]
        else:
            X = [np.concat([rgb_list[ii], nrg_list[ii]]) for ii in range(len(rgb_list))]
            X_train, X_val, self.X_test = [X[ii] for ii in train_idx], [X[ii] for ii in val_idx], [X[ii] for ii in test_idx]
            y_train, y_val, self.y_test = [raw_masks_list[ii] for ii in train_idx], [raw_masks_list[ii] for ii in val_idx], [raw_masks_list[ii] for ii in test_idx]

        aug_factor = 1
        if use_reflection and use_rotation:
            aug_factor = 8
        elif use_reflection or use_rotation:
            aug_factor = 4

        aug_train_idx = [(ii, jj == 0) for ii in train_idx for jj in range(aug_factor)]
        aug_val_idx = [(ii, jj == 0) for ii in val_idx for jj in range(aug_factor)]

        X_train_boot, y_train_boot = md.bootstrap_data(X_train, y_train, rotation=use_rotation, mirroring=use_reflection, return_arr=size_std_strat != '')
        X_val_boot, y_val_boot = md.bootstrap_data(X_val, y_val, rotation=use_rotation, mirroring=use_reflection, return_arr=size_std_strat != '')

        # train_pixel_count, val_pixel_count = sum([x.size for x in y_train_boot]), sum([x.size for x in y_val_boot])

        if size_std_strat != '':
            train_dataset = md.SquareDataset(X_train_boot, y_train_boot, aug_train_idx)
            val_dataset = md.SquareDataset(X_val_boot, y_val_boot, aug_val_idx)
        else:
            train_dataset = md.VariableDataset(X_train_boot, y_train_boot)
            val_dataset = md.VariableDataset(X_val_boot, y_val_boot)

        batch_size = 8 if size_std_dim <= 256 or size_std_strat == 'zero_padding' else 4
        self.train_loader = DataLoader(
            train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers
        )

        self.val_loader = DataLoader(
            val_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers
        )

        self.train_idx, self.val_idx, self.test_idx = train_idx, val_idx, test_idx
        self.metadata_df = metadata_df
        self.raw_masks_list, self.masks_list = raw_masks_list, masks_list

hyperparams = [
     {'model_name': 'ada-net-adapt', 'size_std_strat': 'use_resize', 'size_std_dim': 224, 'use_rotation': False, 'use_reflection': False},
     {'model_name': 'ada-net-adapt', 'size_std_strat': 'use_resize', 'size_std_dim': 224, 'use_rotation': True, 'use_reflection': True},
     {'model_name': 'ada-net-adapt', 'size_std_strat': 'use_resize', 'size_std_dim': 224, 'use_rotation': True, 'use_reflection': True},
    # {'model_name': 'u-net', 'size_std_strat': 'zero_padding', 'size_std_dim': 660, 'use_rotation': False, 'use_reflection': False},
    # {'model_name': 'u-net', 'size_std_strat': 'zero_padding', 'size_std_dim': 660, 'use_rotation': True, 'use_reflection': False},
    #{'model_name': 'u-net', 'size_std_strat': 'use_resize', 'size_std_dim': 256, 'use_rotation': False, 'use_reflection': False},
    #{'model_name': 'u-net', 'size_std_strat': 'use_resize', 'size_std_dim': 256, 'use_rotation': True, 'use_reflection': False},
   # {'model_name': 'u-net', 'size_std_strat': 'use_resize', 'size_std_dim': 384, 'use_rotation': True, 'use_reflection': False},
    #{'model_name': 'u-net', 'size_std_strat': 'use_resize', 'size_std_dim': 128, 'use_rotation': True, 'use_reflection': False},
    #{'model_name': 'u-net', 'size_std_strat': 'use_patching', 'size_std_dim': 64, 'use_rotation': False, 'use_reflection': False},
    #{'model_name': 'u-net', 'size_std_strat': 'use_patching', 'size_std_dim': 64, 'use_rotation': True, 'use_reflection': False},
    #{'model_name': 'u-net', 'size_std_strat': 'use_patching', 'size_std_dim': 128, 'use_rotation': True, 'use_reflection': False}
]


def main(train_model):
    for ii, params in enumerate(hyperparams):

        torch.cuda.empty_cache()
        gc.collect()

        logging.info(f'{ii + 1} / {len(hyperparams)}')
        logging.info(params)
        logging.info('Loading Data')
        rot_str = 'use_rot' if params['use_rotation'] else 'no_rot'
        refl_str = 'use_refl' if params['use_reflection'] else 'no_refl'
        model_group = f'{params["model_name"]}_{params["size_std_strat"]}_{params["size_std_dim"]}_{rot_str}_{refl_str}'
        results_df = pd.read_parquet(results_fp)

        if len(results_df.columns) > 0 and model_group in results_df['model_group'].values: continue

        new_results_row = {key: val for key, val in params.items()}
        new_timing_row = {key: val for key, val in params.items()}
        ts = str(dt.now())[:-7].replace(':', '_').replace('-', '_').replace(' ', '-')
        new_timing_row['timestamp'] = ts
        new_timing_row['model_group'] = model_group
        new_results_row['timestamp'] = ts
        new_results_row['model_group'] = model_group

        training_start = time.time()

        data = DataPreprocessing(params['model_name'], params['size_std_strat'], params['size_std_dim'], params['use_rotation'], params['use_reflection'])

        if params['model_name'] == 'u-net':
            model = md.UNet(in_channels=6, out_channels=1).to(device)
        elif params['model_name'] == 'ada-net-adapt':
            model = md.ADA_single_domain().to(device)

        adam_opt = torch.optim.Adam(model.parameters(), lr=1e-4)

        pos_pixels = data.metadata_df['num_target_pixel'].sum()
        pos_ratio = (sum([x.size for x in data.masks_list]) - pos_pixels) / pos_pixels

        bce_loss = nn.BCEWithLogitsLoss(reduction='sum', pos_weight=torch.tensor(pos_ratio))
        jaccard_loss = smp.losses.JaccardLoss(mode='binary', from_logits=True)

        def combined_loss(bce_loss, jaccard_loss, pred, true, weight):
            return len(true) * ((weight * bce_loss(pred, true)) / pred.numel() + (1 - weight) * jaccard_loss(pred, true))

        train_loss_list = []
        val_loss_list = []

        models_list = [x for x in sorted(os.listdir(model_dir), reverse=True) if x.startswith(model_group)]

        if not train_model and len(models_list) > 0:
            logging.info('Loading Model')
            model = torch.load(os.path.join(model_dir, models_list[0]), weights_only=False)

        else:
            logging.info('Training Model')
            best_model = copy.deepcopy(model.state_dict())
            best_val_perf = np.inf

            min_epochs = 20 if len(data.train_loader.dataset) < 5000 else 8

            # TRAINING
            for epoch in range(num_epochs):
                model.train()
                train_loss_list.append(0)
                val_loss_list.append(0)
                logging.info(f'Epoch: {epoch}, {"Last Epoch Val. Loss: " + str(round(val_loss_list[-2], 4)) if epoch > 0 else ""}')
                for ii, (X, y, _) in enumerate(tqdm(data.train_loader)):
                    X, y = X.to(device), y.to(device)
                    preds = model(X).reshape(y.shape)
                    adam_opt.zero_grad()
                    loss = combined_loss(bce_loss, jaccard_loss, preds, y, weight=combined_loss_weight)
                    loss.backward()
                    adam_opt.step()
                    train_loss_list[-1] += loss.item()
                del X, y, preds
                train_loss_list[-1] /= len(data.train_loader.dataset)

                # Validation Step
                model.eval()
                with torch.no_grad():
                    for X, y, _ in data.val_loader:
                        X, y = X.to(device), y.to(device)
                        preds = model(X).reshape(y.shape)
                        loss = combined_loss(bce_loss, jaccard_loss, preds, y, weight=combined_loss_weight)
                        val_loss_list[-1] += loss.item()
                del X, y, preds
                val_loss_list[-1] /= len(data.val_loader.dataset)
                torch.cuda.empty_cache()
                gc.collect()

                # Update best model if we beat results on validation set
                if val_loss_list[-1] < best_val_perf:
                    best_model = copy.deepcopy(model.state_dict())
                    best_val_perf = val_loss_list[-1]

                # Early Stopping
                if epoch > min_epochs and md.check_stopping_tol(val_loss_list, patience, tol_neg_sig):
                    break

            model.load_state_dict(best_model)
            torch.save(model, os.path.join(model_dir, f'{model_group}_{ts}.pt'))

            new_timing_row['training_time'] = time.time() - training_start


        ### Testing
        logging.info('Testing')
        testing_time = time.time()

        # First we look at preds across train and validation to fit optimum decision threshold
        if params['size_std_strat'] == 'use_resize':
            train_iou_list, threshholds, resized_train_iou_list = md.get_train_iou_list(model, device, data.train_loader, data.val_loader, data.masks_list, data.raw_masks_list, use_resize=True)
        elif params['size_std_strat'] == 'use_patching':
            train_iou_list, threshholds, _ = md.get_train_iou_list(model, device, data.train_loader, data.val_loader, data.masks_list, use_resize=False)

        opt_cutoff = threshholds[np.argmax(train_iou_list)]
        if params['size_std_strat'] == 'use_resize':
            new_results_row['train_iou'] = max(resized_train_iou_list)

        if params['size_std_strat'] != '':
            test_dataset = md.SquareDataset(data.X_test, data.y_test, data.test_idx)
        else:
            test_dataset = md.VariableDataset(data.X_test, data.y_test)

        batch_size = 8 if params['size_std_dim'] <= 256 or params['size_std_strat'] == 'zero_padding' else 4
        test_loader = DataLoader(
            test_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers
        )

        if params['size_std_strat'] == 'use_resize':
            test_iou, threshholds, resized_test_iou = md.get_test_iou_list(model, device, test_loader, opt_cutoff, data.masks_list, data.raw_masks_list, use_resize=True)
            new_results_row['resized_test_iou'] = resized_test_iou
        elif params['size_std_strat'] == 'use_patching':
            test_iou, threshholds, _ = md.get_test_iou_list(model, device, test_loader, opt_cutoff, data.masks_list, [], use_resize=False)

        new_timing_row['testing_time'] = time.time() - testing_time
        new_results_row['test_iou'] = test_iou
        new_results_row['train_iou'] = max(train_iou_list)
        new_results_row['opt_cutoff'] = opt_cutoff
        new_results_row['epochs'] = len(train_loss_list)

        # Add timing record (update at the end of each run so no progress is lost)
        pd.concat([pd.read_parquet(timing_fp), pd.DataFrame([new_timing_row])]).to_parquet(timing_fp)

        # Add results record (update at the end of each run so no progress is lost)
        pd.concat([results_df, pd.DataFrame([new_results_row])]).to_parquet(results_fp)

        # Clear out everything
        del data, test_dataset, test_loader, model
        torch.cuda.empty_cache()
        gc.collect()


if __name__ == '__main__':
    main(train_model=False)

INFO - 2025-08-04 00:37:08,779 - 1 / 3
INFO - 2025-08-04 00:37:08,783 - {'model_name': 'ada-net-adapt', 'size_std_strat': 'use_resize', 'size_std_dim': 224, 'use_rotation': False, 'use_reflection': False}
INFO - 2025-08-04 00:37:08,783 - Loading Data
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to C:\Users\Angel/.cache\torch\hub\checkpoints\resnet50-11ad3fa6.pth
100%|██████████| 97.8M/97.8M [00:06<00:00, 15.5MB/s]


RuntimeError: Found no NVIDIA driver on your system. Please check that you have an NVIDIA GPU and installed a driver from http://www.nvidia.com/Download/index.aspx